# Caso 03: Predicción de Fuga de Clientes (Churn) - AndinaTel S.A.C.
Pipeline end-to-end de aprendizaje supervisado de clasificación.

1. Generar e inspeccionar el dataset crudo.
2. Limpiar, codificar y dividir los datos (80/20 estratificado).
3. Construir y evaluar modelos (LR, KNN, SVM, RF, GB).
4. Seleccionar el modelo óptimo y optimizar.
5. Con el modelo optimizado, realizar la predicción para nuevos clientes.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

## Fase 1: Generación e inspección del dataset

In [2]:
ruta_caso = Path.cwd()
if not (ruta_caso / 'generar_dataset_andinatel.py').exists():
    ruta_caso = ruta_caso / 'aprendizaje_supervisado' / 'clasificacion' / 'caso_03'
if not (ruta_caso / 'clientes_andinatel_raw.csv').exists():
    import subprocess, sys
    subprocess.run([sys.executable, str(ruta_caso / 'generar_dataset_andinatel.py')], check=True)
df_data = pd.read_csv(ruta_caso / 'clientes_andinatel_raw.csv')

In [3]:
df_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID_Cliente           1250 non-null   str    
 1   Fecha_Alta           1250 non-null   str    
 2   Plan_Tarifario       1250 non-null   str    
 3   Cargo_Mensual        1199 non-null   str    
 4   Consumo_Datos_GB     1250 non-null   str    
 5   Reclamos_6M          1226 non-null   str    
 6   Metodo_Pago          1250 non-null   str    
 7   Tipo_Contrato        1250 non-null   str    
 8   Score_Crediticio     1221 non-null   float64
 9   Soporte_Prioritario  1250 non-null   str    
 10  Fuga_Cliente         1250 non-null   str    
dtypes: float64(1), str(10)
memory usage: 107.6 KB


In [4]:
df_data.isna().sum()

ID_Cliente              0
Fecha_Alta              0
Plan_Tarifario          0
Cargo_Mensual          51
Consumo_Datos_GB        0
Reclamos_6M            24
Metodo_Pago             0
Tipo_Contrato           0
Score_Crediticio       29
Soporte_Prioritario     0
Fuga_Cliente            0
dtype: int64

In [5]:
df_data.head(10)

,ID_Cliente,Fecha_Alta,Plan_Tarifario,Cargo_Mensual,Consumo_Datos_GB,Reclamos_6M,Metodo_Pago,Tipo_Contrato,Score_Crediticio,Soporte_Prioritario,Fuga_Cliente
0,CLI-00001,18/09/2024,Intermedio,S/78.82,42.4 GB,1,App_Bancaria,Anual_12M,510.0,No,0
1,CLI-00002,2025-03-30,Ultra,S/187.18,57.6 GB,3.0,Debito_Automatico,Mensual,-999.0,No,1
2,CLI-00003,2023-07-14,Premium,S/ 111.78,32.6 GB,ninguno,Agente_Efectivo,Mensual,490.0,No,Si
3,CLI-00004,2025-01-02,Intermedio,S/ 101.51,50.5 GB,5,Debito_Automatico,Mensual,800.0,0,1
4,CLI-00005,08/02/2026,Basico,S/ 49.42,37.1 GB,2,App_Bancaria,Mensual,726.0,True,No
5,CLI-00006,2024-01-09,Basico,S/ 32.73,16.2 GB,0,Debito_Automatico,Anual_12M,466.0,No,0
6,CLI-00007,30/12/2020,Basico,S/51.44,46.4 GB,2.0,App_Bancaria,Mensual,653.0,0,0
7,CLI-00008,08/04/2021,Premium,S/ 111.73,31.8 GB,3,App_Bancaria,Mensual,569.0,1,1
8,CLI-00009,04/10/2022,Intermedio,S/89.99,49.0 GB,1,Debito_Automatico,Mensual,584.0,0,0
9,CLI-00010,06/11/2020,Premium,S/112.07,73.0 GB,ninguno,Debito_Automatico,Bianual_24M,652.0,0,No


## Fase 2: Limpieza y partición de datos
### 1. Carga y saneamiento de tipos

In [6]:
fecha_corte = pd.Timestamp('2026-11-01')

def convertir_fecha(valor):
    # Fechas mixtas: '2023-05-14' (ISO) y '14/05/2023' (dia/mes/anio)
    if re.match(r'^\d{4}-\d{2}-\d{2}$', str(valor)):
        return pd.to_datetime(valor, format='%Y-%m-%d')
    return pd.to_datetime(valor, format='%d/%m/%Y')

def limpiar_numero(valor):
    # Extrae el numero de textos como 'S/ 89.90', 'S/120.50' o '45.2 GB'; 'N/D' -> NaN
    encontrado = re.search(r'\d+(\.\d+)?', str(valor))
    return float(encontrado.group()) if encontrado else np.nan

def limpiar_reclamos(valor):
    if pd.isna(valor):
        return np.nan
    if str(valor).strip().lower() == 'ninguno':
        return 0.0
    return round(float(valor))

def a_binario(valor):
    mapa = {'si': 1, '1': 1, 'true': 1, 'no': 0, '0': 0, 'false': 0}
    return mapa[str(valor).strip().lower()]

In [7]:
def limpiar_tipos(df):
    df = df.copy()
    df['Antiguedad_Dias'] = (fecha_corte - df['Fecha_Alta'].apply(convertir_fecha)).dt.days.astype('int64')
    df['Cargo_Mensual'] = df['Cargo_Mensual'].apply(limpiar_numero).astype('float64')
    df['Consumo_Datos_GB'] = df['Consumo_Datos_GB'].apply(limpiar_numero).astype('float64')
    df['Reclamos_6M'] = df['Reclamos_6M'].apply(limpiar_reclamos).astype('float64')
    df['Score_Crediticio'] = pd.to_numeric(df['Score_Crediticio'], errors='coerce').astype('float64')
    df['Soporte_Prioritario'] = df['Soporte_Prioritario'].apply(a_binario).astype('int64')
    if 'Fuga_Cliente' in df.columns:
        df['Fuga_Cliente'] = df['Fuga_Cliente'].apply(a_binario).astype('int64')
    return df

df_data = limpiar_tipos(df_data)
df_data.head()

,ID_Cliente,Fecha_Alta,Plan_Tarifario,Cargo_Mensual,Consumo_Datos_GB,Reclamos_6M,Metodo_Pago,Tipo_Contrato,Score_Crediticio,Soporte_Prioritario,Fuga_Cliente,Antiguedad_Dias
0,CLI-00001,18/09/2024,Intermedio,78.82,42.4,1.0,App_Bancaria,Anual_12M,510.0,0,0,774
1,CLI-00002,2025-03-30,Ultra,187.18,57.6,3.0,Debito_Automatico,Mensual,-999.0,0,1,581
2,CLI-00003,2023-07-14,Premium,111.78,32.6,0.0,Agente_Efectivo,Mensual,490.0,0,1,1206
3,CLI-00004,2025-01-02,Intermedio,101.51,50.5,5.0,Debito_Automatico,Mensual,800.0,0,1,668
4,CLI-00005,08/02/2026,Basico,49.42,37.1,2.0,App_Bancaria,Mensual,726.0,1,0,266


### 2. Tratamiento de anomalías y nulos
El código `-999` en `Score_Crediticio` se trata como nulo y los nulos numéricos se imputan con la mediana.

In [8]:
df_data['Score_Crediticio'] = df_data['Score_Crediticio'].replace(-999, np.nan)

columnas_numericas = ['Cargo_Mensual', 'Consumo_Datos_GB', 'Reclamos_6M', 'Score_Crediticio']
medianas = df_data[columnas_numericas].median()
df_data[columnas_numericas] = df_data[columnas_numericas].fillna(medianas)
df_data['Reclamos_6M'] = df_data['Reclamos_6M'].astype('int64')
print(medianas)
df_data.isna().sum()

Cargo_Mensual        80.93
Consumo_Datos_GB     31.50
Reclamos_6M           1.00
Score_Crediticio    617.00
dtype: float64


ID_Cliente             0
Fecha_Alta             0
Plan_Tarifario         0
Cargo_Mensual          0
Consumo_Datos_GB       0
Reclamos_6M            0
Metodo_Pago            0
Tipo_Contrato          0
Score_Crediticio       0
Soporte_Prioritario    0
Fuga_Cliente           0
Antiguedad_Dias        0
dtype: int64

### 3. Codificación de variables categóricas

In [9]:
mapa_plan = {'Basico': 0, 'Intermedio': 1, 'Premium': 2, 'Ultra': 3}

def codificar(df):
    df = df.drop(['ID_Cliente', 'Fecha_Alta'], axis=1, errors='ignore')
    df['Plan_Tarifario'] = df['Plan_Tarifario'].map(mapa_plan).astype('int64')
    df = pd.get_dummies(df, columns=['Metodo_Pago', 'Tipo_Contrato'], drop_first=True, dtype='uint8')
    return df

df_data = codificar(df_data)
df_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Plan_Tarifario                 1250 non-null   int64  
 1   Cargo_Mensual                  1250 non-null   float64
 2   Consumo_Datos_GB               1250 non-null   float64
 3   Reclamos_6M                    1250 non-null   int64  
 4   Score_Crediticio               1250 non-null   float64
 5   Soporte_Prioritario            1250 non-null   int64  
 6   Fuga_Cliente                   1250 non-null   int64  
 7   Antiguedad_Dias                1250 non-null   int64  
 8   Metodo_Pago_App_Bancaria       1250 non-null   uint8  
 9   Metodo_Pago_Debito_Automatico  1250 non-null   uint8  
 10  Tipo_Contrato_Bianual_24M      1250 non-null   uint8  
 11  Tipo_Contrato_Mensual          1250 non-null   uint8  
dtypes: float64(3), int64(5), uint8(4)
memory usage: 83.1 KB


### 4. Separación y división estratificada

In [10]:
df_X = df_data.drop('Fuga_Cliente', axis=1)
X = df_X.values
y = df_data['Fuga_Cliente'].values
print(df_data['Fuga_Cliente'].value_counts(normalize=True))

Fuga_Cliente
0    0.7312
1    0.2688
Name: proportion, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)
print(X_train.shape, X_test.shape)

(1000, 11) (250, 11)


## Fase 3: Comparativa de algoritmos de clasificación
Cada modelo va en un pipeline con `StandardScaler` y se evalúa con `StratifiedKFold(n_splits=5)` solo sobre `X_train`.

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = [
    ('LR', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
    ('KNN', make_pipeline(StandardScaler(), KNeighborsClassifier())),
    ('SVM', make_pipeline(StandardScaler(), SVC(random_state=42))),   # roc_auc usa decision_function
    ('RF', make_pipeline(StandardScaler(), RandomForestClassifier(random_state=42))),
    ('GB', make_pipeline(StandardScaler(), GradientBoostingClassifier(random_state=42))),
]
metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
result = []
names = []
for name, model in models:
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_results = cross_validate(model, X_train, y_train, cv=kfold, scoring=metricas)
    result.append({m: cv_results['test_' + m].mean() for m in metricas})
    names.append(name)
    msg = "%s: ROC-AUC %f (%f) | F1 %f" % (name, cv_results['test_roc_auc'].mean(), cv_results['test_roc_auc'].std(), cv_results['test_f1'].mean())
    print(msg)

LR: ROC-AUC 0.792203 (0.009051) | F1 0.468673
KNN: ROC-AUC 0.714190 (0.004431) | F1 0.435644
SVM: ROC-AUC 0.763749 (0.019693) | F1 0.381066
RF: ROC-AUC 0.743923 (0.026561) | F1 0.386364
GB: ROC-AUC 0.750606 (0.020173) | F1 0.374328


In [13]:
df_comparativa = pd.DataFrame(result, index=names).round(4)
df_comparativa = df_comparativa.sort_values(['roc_auc', 'f1'], ascending=False)
df_comparativa

,accuracy,precision,recall,f1,roc_auc
LR,0.766,0.6045,0.3869,0.4687,0.7922
SVM,0.756,0.5928,0.2867,0.3811,0.7637
GB,0.735,0.5031,0.3092,0.3743,0.7506
RF,0.742,0.5401,0.3128,0.3864,0.7439
KNN,0.749,0.5591,0.3605,0.4356,0.7142


In [14]:
mejor_nombre = df_comparativa.index[0]
print('Modelo ganador:', mejor_nombre)
print(df_comparativa.loc[mejor_nombre])

Modelo ganador: LR
accuracy     0.7660
precision    0.6045
recall       0.3869
f1           0.4687
roc_auc      0.7922
Name: LR, dtype: float64


**Justificación:** con ~27% de fuga, un modelo que diga siempre "no se va" tendría ~73% de Accuracy sin detectar a nadie.
Por eso se prioriza **ROC-AUC** (capacidad de ordenar a los clientes por riesgo, independiente del umbral) y luego **F1-Score**
(equilibrio entre Precision y Recall de la clase fuga). El modelo ganador es el primero de la tabla ordenada por estas dos métricas.

En este dataset gana **Regresión Logística (LR)**: tiene el mayor ROC-AUC (~0.79) y el mayor F1 (~0.47). Las relaciones entre las variables
y la fuga son principalmente lineales (contrato mensual, cargo alto, reclamos), por eso un modelo lineal generaliza mejor que los de árboles.

## Fase 4: Optimización del modelo ganador
`GridSearchCV` usando solo `X_train` y optimizando `roc_auc`.

In [15]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    'LR': {'logisticregression__C': [0.01, 0.1, 1, 10, 100],
           'logisticregression__class_weight': [None, 'balanced']},
    'KNN': {'kneighborsclassifier__n_neighbors': list(range(3, 51, 2)),
            'kneighborsclassifier__weights': ['uniform', 'distance']},
    'SVM': {'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 0.01, 0.1],
            'svc__class_weight': [None, 'balanced']},
    'RF': {'randomforestclassifier__n_estimators': [100, 300],
           'randomforestclassifier__max_depth': [None, 5, 10],
           'randomforestclassifier__min_samples_leaf': [1, 5],
           'randomforestclassifier__class_weight': [None, 'balanced']},
    'GB': {'gradientboostingclassifier__n_estimators': [100, 200],
           'gradientboostingclassifier__learning_rate': [0.05, 0.1],
           'gradientboostingclassifier__max_depth': [2, 3, 4]},
}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
modelo_base = dict(models)[mejor_nombre]
if mejor_nombre == 'SVM':
    # SVC necesita probabilidades para la Fase 5
    from sklearn.calibration import CalibratedClassifierCV
    modelo_base = make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(random_state=42), ensemble=False))
    param_grids['SVM'] = {'calibratedclassifiercv__estimator__' + k.split('__')[1]: v for k, v in param_grids['SVM'].items()}
grid_search = GridSearchCV(modelo_base, param_grids[mejor_nombre], cv=kfold, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

{'logisticregression__C': 10, 'logisticregression__class_weight': 'balanced'}
0.79393064080691


In [16]:
print('ROC-AUC base (sin optimizar): %f' % df_comparativa.loc[mejor_nombre, 'roc_auc'])
print('ROC-AUC optimizado        : %f' % grid_search.best_score_)
print('Mejora                    : %+f' % (grid_search.best_score_ - df_comparativa.loc[mejor_nombre, 'roc_auc']))

ROC-AUC base (sin optimizar): 0.792200
ROC-AUC optimizado        : 0.793931
Mejora                    : +0.001731


### Evaluación final sobre el 20% de prueba

In [17]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

modelo_final = grid_search.best_estimator_
y_pred = modelo_final.predict(X_test)
y_proba = modelo_final.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred, target_names=['Permanece (0)', 'Fuga (1)']))
print('ROC-AUC en test: %f' % roc_auc_score(y_test, y_proba))

               precision    recall  f1-score   support

Permanece (0)       0.90      0.71      0.79       183
     Fuga (1)       0.50      0.78      0.60        67

     accuracy                           0.73       250
    macro avg       0.70      0.74      0.70       250
 weighted avg       0.79      0.73      0.74       250

ROC-AUC en test: 0.830275


In [18]:
pd.DataFrame(confusion_matrix(y_test, y_pred),
             index=['Real: Permanece', 'Real: Fuga'],
             columns=['Pred: Permanece', 'Pred: Fuga'])

,Pred: Permanece,Pred: Fuga
Real: Permanece,130,53
Real: Fuga,15,52


### Complemento (Imbalanced-Learn): modelo ganador con SMOTE
SMOTE se aplica dentro del pipeline de `imblearn`, así solo genera datos sintéticos en cada partición de entrenamiento.

In [20]:
from imblearn.pipeline import make_pipeline as make_pipeline_imb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score

modelo_smote = make_pipeline_imb(StandardScaler(), SMOTE(random_state=42), modelo_final.steps[-1][1])
cv_smote = cross_val_score(modelo_smote, X_train, y_train, cv=kfold, scoring='roc_auc')
cv_smote_f1 = cross_val_score(modelo_smote, X_train, y_train, cv=kfold, scoring='f1')
print('Con SMOTE -> ROC-AUC: %f (%f) | F1: %f' % (cv_smote.mean(), cv_smote.std(), cv_smote_f1.mean()))

Con SMOTE -> ROC-AUC: 0.792927 (0.013952) | F1: 0.568451


## Fase 5: Inferencia sobre nuevos clientes
Tres prospectos con perfiles de riesgo alto, moderado y bajo. Pasan por la misma limpieza y codificación, y sus columnas se alinean con `df_X.columns`.

In [21]:
df_new = pd.DataFrame({
    'ID_Cliente': ['NEW-00001', 'NEW-00002', 'NEW-00003'],
    'Fecha_Alta': ['2026-08-15', '10/01/2025', '2021-03-20'],
    'Plan_Tarifario': ['Ultra', 'Intermedio', 'Premium'],
    'Cargo_Mensual': ['S/ 185.90', 'S/95.00', 'S/ 115.50'],
    'Consumo_Datos_GB': ['12.5 GB', 'N/D', '60.3 GB'],
    'Reclamos_6M': ['5', '2.0', 'ninguno'],
    'Metodo_Pago': ['Agente_Efectivo', 'App_Bancaria', 'Debito_Automatico'],
    'Tipo_Contrato': ['Mensual', 'Anual_12M', 'Bianual_24M'],
    'Score_Crediticio': [-999, 610, 780],
    'Soporte_Prioritario': ['No', '0', 'Si'],
})
df_new = limpiar_tipos(df_new)
df_new['Score_Crediticio'] = df_new['Score_Crediticio'].replace(-999, np.nan)
df_new[columnas_numericas] = df_new[columnas_numericas].fillna(medianas)   # medianas del entrenamiento
df_new['Reclamos_6M'] = df_new['Reclamos_6M'].astype('int64')
ids_new = df_new['ID_Cliente']
df_X_new = codificar(df_new).reindex(columns=df_X.columns, fill_value=0).astype(df_X.dtypes.to_dict())
assert list(df_X_new.columns) == list(df_X.columns)
X_new = df_X_new.values
df_X_new

,Plan_Tarifario,Cargo_Mensual,Consumo_Datos_GB,Reclamos_6M,Score_Crediticio,Soporte_Prioritario,Antiguedad_Dias,Metodo_Pago_App_Bancaria,Metodo_Pago_Debito_Automatico,Tipo_Contrato_Bianual_24M,Tipo_Contrato_Mensual
0,3,185.9,12.5,5,617.0,0,78,0,0,0,1
1,1,95.0,31.5,2,610.0,0,660,1,0,0,0
2,2,115.5,60.3,0,780.0,1,2052,0,1,1,0


In [22]:
y_prediction = modelo_final.predict(X_new)
y_probabilidad = modelo_final.predict_proba(X_new)[:, 1]
print("Predicciones: {}".format(y_prediction))
print("Probabilidad de fuga: {}".format(np.round(y_probabilidad, 4)))

Predicciones: [1 1 0]
Probabilidad de fuga: [0.9945 0.5719 0.0091]


In [23]:
def regla_negocio(probabilidad):
    if probabilidad >= 0.65:
        return 'Riesgo Critico', 'Llamada inmediata del equipo de retencion + descuento o migracion a contrato anual'
    elif probabilidad >= 0.35:
        return 'Riesgo Moderado', 'Campana preventiva: beneficio de datos extra y encuesta de satisfaccion'
    return 'Riesgo Bajo', 'Seguimiento estandar y ofertas de fidelizacion'

df_resultado = pd.DataFrame({
    'ID_Cliente': ids_new,
    'Prediccion': y_prediction,
    'Prob_Fuga': np.round(y_probabilidad * 100, 2),
})
df_resultado[['Nivel_Alerta', 'Accion_Retencion']] = [regla_negocio(p) for p in y_probabilidad]
df_resultado

,ID_Cliente,Prediccion,Prob_Fuga,Nivel_Alerta,Accion_Retencion
0,NEW-00001,1,99.45,Riesgo Critico,Llamada inmediata del equipo de retencion + de...
1,NEW-00002,1,57.19,Riesgo Moderado,Campana preventiva: beneficio de datos extra y...
2,NEW-00003,0,0.91,Riesgo Bajo,Seguimiento estandar y ofertas de fidelizacion


## Conclusiones
- El modelo optimizado usa `class_weight='balanced'`, lo que sube mucho el **Recall** de la clase fuga: detecta a la mayoría de los clientes que se van, a cambio de algunas falsas alarmas.
- Para AndinaTel esto conviene: una falsa alarma cuesta una llamada de retención, mientras que perder un cliente cuesta 5 veces más que retenerlo.
- SMOTE da un resultado parecido a `class_weight='balanced'`, por eso el modelo final se quedó con la opción más simple.
- La regla de negocio transforma la probabilidad en un nivel de alerta y una acción concreta para el área comercial.